# Automatic Ticket Classification & Topic Modeling\n
\n
## Problem Statement\n
You need to build a model that is able to classify customer complaints based on the products/services. By doing so, you can segregate these tickets into their relevant categories and, therefore, help in the quick resolution of the issue.\n
\n
You will be doing topic modelling on the **.json** data provided by the company. Since this data is not labelled, you need to apply NMF to analyse patterns and classify tickets into the following five clusters based on their products/services:\n
- **Credit card / Prepaid card**\n
- **Bank account services**\n
- **Theft/Dispute reporting**\n
- **Mortgages/loans**\n
- **Others**\n
\n
With the help of topic modelling, you will be able to map each ticket onto its respective department/category. You can then use this data to train any supervised model such as logistic regression, decision tree or random forest. Using this trained model, you can classify any new customer complaint support ticket into its relevant department.

## Pipelines that need to be performed:\n
1. Data loading\n
2. Text preprocessing\n
3. Exploratory data analysis (EDA)\n
4. Feature extraction\n
5. Topic modelling (NMF)\n
6. Model building using supervised learning\n
7. Model training and evaluation\n
8. Model inference

## Importing the necessary libraries

In [ ]:
import os\n
import json\n
import numpy as np\n
import pandas as pd\n
import re, nltk, spacy, string\n
import en_core_web_sm\n
nlp = en_core_web_sm.load()\n
import seaborn as sns\n
import matplotlib.pyplot as plt\n
%matplotlib inline\n
import warnings\n
warnings.filterwarnings('ignore')\n
\n
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer\n
from sklearn.decomposition import NMF\n
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split\n
from sklearn.linear_model import LogisticRegression\n
from sklearn.tree import DecisionTreeClassifier\n
from sklearn.ensemble import RandomForestClassifier\n
from sklearn.naive_bayes import MultinomialNB\n
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay\n
from tqdm import tqdm\n
tqdm.pandas()\n
\n
# Display settings\n
pd.set_option('display.max_colwidth', None)\n
pd.set_option('display.max_columns', None)

## 1. Loading the data\n
The data is in JSON format and we convert it to a pandas dataframe.

In [ ]:
# Load JSON or fallback preprocessed dataset\n
json_path = 'complaints-2021-05-14_08_16.json'\n
if os.path.exists(json_path):\n
    with open(json_path) as f:\n
        data = json.load(f)\n
    df = pd.json_normalize(data)\n
    print('Columns in dataset:', df.columns.values)\n
    \n
    # Rename columns\n
    df.rename(columns={\n
        '_index': 'index', '_type': 'type', '_id': 'id', '_score': 'score',\n
        '_source.tags': 'tags', '_source.zip_code': 'zip_code', '_source.complaint_id': 'complaint_id',\n
        '_source.issue': 'issue', '_source.date_received': 'date_received', '_source.state': 'state',\n
        '_source.consumer_disputed': 'consumer_disputed', '_source.product': 'product',\n
        '_source.company_response': 'company_response', '_source.company': 'company',\n
        '_source.submitted_via': 'submitted_via', '_source.date_sent_to_company': 'date_sent_to_company',\n
        '_source.company_public_response': 'company_public_response', '_source.sub_product': 'sub_product',\n
        '_source.timely': 'timely', '_source.complaint_what_happened': 'complaint_what_happened',\n
        '_source.sub_issue': 'sub_issue', '_source.consumer_consent_provided': 'consumer_consent_provided'\n
    }, inplace=True)\n
    \n
    df['complaint_what_happened'].replace('', np.nan, inplace=True)\n
    df.dropna(subset=['complaint_what_happened'], inplace=True)\n
else:\n
    print('Using local cleaned tickets data...')\n
    df = pd.read_csv('model/output/cleaned_tickets.csv')\n
\n
df.head()

## 2. Text Preprocessing\n
- Lowercase conversion\n
- Remove text in square brackets\n
- Remove punctuation\n
- Remove words with numbers\n
- Lemmatization with stop words removal\n
- POS tag filtering for NN nouns only\n
- Remove -PRON- and masked personal details (xxxx)

In [ ]:
def clean_text(text):\n
    if pd.isna(text): return ''\n
    text = str(text).lower()\n
    text = re.sub(r'\[.*?\]', ' ', text)\n
    text = re.sub(r'[^\w\s]', ' ', text)\n
    text = re.sub(r'\w*\d\w*', ' ', text)\n
    text = re.sub(r'\s+', ' ', text).strip()\n
    return text\n
\n
stopwords = nlp.Defaults.stop_words\n
\n
def lemmatization(texts):\n
    lemma_sentences = []\n
    for doc in tqdm(nlp.pipe(texts, batch_size=256)):\n
        sent = [token.lemma_ for token in doc if token.text not in stopwords and len(token.lemma_) > 1]\n
        lemma_sentences.append(' '.join(sent))\n
    return lemma_sentences\n
\n
def extract_pos_tags(texts):\n
    pos_sentences = []\n
    for doc in tqdm(nlp.pipe(texts, batch_size=256)):\n
        sent = [token.text for token in doc if token.tag_ == 'NN']\n
        pos_sentences.append(' '.join(sent))\n
    return pos_sentences\n
\n
df_clean = pd.DataFrame()\n
df_clean['complaint_what_happened'] = df['complaint_what_happened'].progress_apply(clean_text)\n
df_clean['complaint_what_happened_lemmatized'] = lemmatization(df_clean['complaint_what_happened'])\n
df_clean['complaint_POS_removed'] = extract_pos_tags(df_clean['complaint_what_happened_lemmatized'])\n
df_clean['Complaint_clean'] = df_clean['complaint_POS_removed'].str.replace('-PRON-', '', regex=False)\n
df_clean['Complaint_clean'] = df_clean['Complaint_clean'].str.replace('xxxx', '', regex=False)\n
df_clean.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
def get_top_n_words(corpus, n=1, count=10):\n
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)\n
    bag_of_words = vec.transform(corpus)\n
    sum_words = bag_of_words.sum(axis=0)\n
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]\n
    return sorted(words_freq, key=lambda x: x[1], reverse=True)[:count]\n
\n
print('Top 10 Unigrams:', get_top_n_words(df_clean['Complaint_clean'], 1, 10))\n
print('Top 10 Bigrams:', get_top_n_words(df_clean['Complaint_clean'], 2, 10))\n
print('Top 10 Trigrams:', get_top_n_words(df_clean['Complaint_clean'], 3, 10))

## 4. Feature Extraction\n
Converting cleaned complaints to TF-IDF matrix (max_df=0.98, min_df=2, stop_words='english').

In [ ]:
tf_idf_vec = TfidfVectorizer(max_df=0.98, min_df=2, stop_words='english')\n
tfidf = tf_idf_vec.fit_transform(df_clean['Complaint_clean'])\n
print('TF-IDF matrix shape:', tfidf.shape)

## 5. Topic Modelling using NMF (5 Topics)\n
Decompose into 5 clusters and map onto products/services:\n
- 0: Bank Account services\n
- 1: Credit card or prepaid card\n
- 2: Others\n
- 3: Theft/Dispute Reporting\n
- 4: Mortgage/Loan

In [ ]:
num_topics = 5\n
nmf_model = NMF(n_components=num_topics, random_state=40)\n
nmf_model.fit(tfidf)\n
\n
for index, topic in enumerate(nmf_model.components_):\n
    print(f'THE TOP 15 WORDS FOR TOPIC #{index}:')\n
    print([tf_idf_vec.get_feature_names_out()[i] for i in topic.argsort()[-15:]])\n
    print('\n')\n
\n
topic_values = nmf_model.transform(tfidf)\n
df_clean['Topic'] = topic_values.argmax(axis=1)\n
\n
Topic_names = {\n
    0: 'Bank Account services',\n
    1: 'Credit card or prepaid card',\n
    2: 'Others',\n
    3: 'Theft/Dispute Reporting',\n
    4: 'Mortgage/Loan'\n
}\n
df_clean['Topic_category'] = df_clean['Topic'].map(Topic_names)\n
df_clean[['complaint_what_happened', 'Topic', 'Topic_category']].head()

## 6 & 7. Supervised Model Building, Training & Evaluation\n
Models evaluated:\n
- Logistic Regression\n
- Decision Tree Classifier\n
- Random Forest Classifier\n
- Multinomial Naive Bayes\n
- XGBoost Classifier

In [ ]:
training_data = df_clean[['complaint_what_happened', 'Topic']].copy()\n
\n
count_vect = CountVectorizer()\n
X_train_counts = count_vect.fit_transform(training_data['complaint_what_happened'])\n
tfidf_transformer = TfidfTransformer()\n
X_train_tf = tfidf_transformer.fit_transform(X_train_counts)\n
\n
train_X, test_X, train_y, test_y = train_test_split(\n
    X_train_tf, training_data['Topic'], test_size=0.2, random_state=40, stratify=training_data['Topic']\n
)\n
\n
def eval_model(y_test, y_pred, y_pred_proba, type_name='Test'):\n
    print(f'{type_name} Results:')\n
    print('Accuracy: ', round(accuracy_score(y_test, y_pred), 2))\n
    print('Precision: ', round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 2))\n
    print('Recall: ', round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 2))\n
    print('F1 Score: ', round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 2))\n
    if y_pred_proba is not None:\n
        try:\n
            print('ROC AUC Score: ', round(roc_auc_score(y_test, y_pred_proba, average='weighted', multi_class='ovr'), 2))\n
        except Exception:\n
            pass\n
    print('Classification Report:\n', classification_report(y_test, y_pred, zero_division=0))\n
\n
# Train and evaluate Logistic Regression\n
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=40, class_weight='balanced')\n
lr_model.fit(train_X, train_y)\n
eval_model(test_y, lr_model.predict(test_X), lr_model.predict_proba(test_X), type_name='Logistic Regression Test')

## 8. Model Inference\n
Classify new customer complaints into their relevant department.

In [ ]:
df_complaints = pd.DataFrame({'complaints': [\n
    "I can not get from chase who services my mortgage, who owns it and who has original loan docs",\n
    "The bill amount of my credit card was debited twice. Please look into the matter and resolve at the earliest.",\n
    "I want to open a salary account at your downtown branch. Please provide me the procedure.",\n
    "Yesterday, I received a fraudulent email regarding renewal of my services.",\n
    "What is the procedure to know my CIBIL score?",\n
    "I need to know the number of bank branches and their locations in the city of Dubai"\n
]})\n
\n
def predict_complaint(text):\n
    X_new_counts = count_vect.transform([text])\n
    X_new_tfidf = tfidf_transformer.transform(X_new_counts)\n
    predicted = lr_model.predict(X_new_tfidf)[0]\n
    return Topic_names[predicted]\n
\n
df_complaints['tag'] = df_complaints['complaints'].apply(predict_complaint)\n
df_complaints